In [1]:
import os
from typing import Annotated, Literal, TypedDict
from datetime import datetime

from langchain_core.tools import tool
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
    RemoveMessage,
)
from langchain_core.runnables import RunnableConfig

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver


import pandas as pd
from requests import get # recupérer le contenu html de la page
from bs4 import BeautifulSoup as bs

In [77]:
from langchain.tools import tool
from requests import get
from bs4 import BeautifulSoup as bs
import re

@tool
def scraping_voitures_tool(pages: int = 1) -> list:
    """Scrape les voitures depuis dakar-auto.com"""

    all_data = []

    headers = {"User-Agent": "Mozilla/5.0"}

    for ind in range(1, pages + 1):
        url = f'https://dakar-auto.com/senegal/voitures-4?&page={ind}'
        res = get(url, headers=headers)
        soup = bs(res.content, 'html.parser')

        containers = soup.find_all('div', 'listings-cards__list-item mb-md-3 mb-3')

        for container in containers:
            try:
                # lien annonce
                link = container.find('a')['href']
                url_container = 'https://dakar-auto.com/' + link

                res_container = get(url_container, headers=headers)
                soup_container = bs(res_container.content, 'html.parser')

                # title
                title = soup_container.find('h1', 'listing-item__title').text.strip()

                # price
                price_text = soup_container.find(
                    'h4',
                    'listing-item__price font-weight-bold text-uppercase mb-2'
                ).text.strip()
                price = int(re.sub(r'\D', '', price_text))

                # kilometrage
                km_text = soup_container.find(
                    'li',
                    'listing-item__attribute list-inline-item'
                ).text.strip()
                kilometrage = int(re.sub(r'\D', '', km_text))

                # bloc info
                bloc = soup_container.find(
                    'ul',
                    'listing-item__attribute-list list-inline'
                ).text.strip().split('\n')

                carburant = bloc[-1]
                boite = bloc[-4]
                annee = int(re.sub(r'\D', '', bloc[3]))

                all_data.append({
                    "title": title,
                    "price": price,
                    "kilometrage": kilometrage,
                    "carburant": carburant,
                    "boite": boite,
                    "annee": annee,
                    "source": url_container
                })

            except Exception as e:
                print("Erreur:", e)

    return all_data
    

In [78]:
data = scraping_voitures_tool.invoke({"pages": 1})

In [ ]:
data

In [80]:
import sqlite3
from langchain_core.tools import tool
from typing import List, Optional

@tool
def manage_car_db(action: str, cars: Optional[List[dict]] = None):
    """
    Gère la base de données SQLite pour les véhicules de dakar-auto.com.
    Actions : 
    - 'init' : Crée la table avec les colonnes (title, price, kilometrage, carburant, boite, annee, source).
    - 'insert' : Sauvegarde une liste de véhicules.
    - 'read' : Récupère tous les véhicules stockés.
    """
    db_path = "annonces_vehicules.db"
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    if action == "init":
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS voitures (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT,
                price INTEGER,
                kilometrage INTEGER,
                carburant INTEGER,
                boite TEXT,
                annee INTEGER,
                source TEXT
            )
        ''')
        conn.commit()
        res = "Base de données et table 'voitures' prêtes."

    elif action == "insert" and cars:
        # Requête paramétrée pour éviter les injections SQL
        query = '''
            INSERT INTO voitures (title, price, kilometrage, carburant, boite, annee, source)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        '''
        # On prépare les données sous forme de tuples
        data_to_insert = [
            (c['title'], c['price'], c['kilometrage'], c['carburant'], c['boite'], c['annee'], c['source'])
            for c in cars
        ]
        
        cursor.executemany(query, data_to_insert)
        conn.commit()
        res = f"Succès : {len(cars)} véhicules insérés dans la base."

    elif action == "read":
        cursor.execute("SELECT * FROM voitures ORDER BY id DESC")
        res = cursor.fetchall()

    else:
        res = "Erreur : Action non reconnue ou aucune donnée fournie."

    conn.close()
    return res


In [71]:
manage_car_db.invoke({"action": "init"})

"Base de données et table 'voitures' prêtes."

In [81]:
resultats_scraping = scraping_voitures_tool.invoke({"pages": 1})

In [ ]:
resultats_scraping

In [84]:
manage_car_db.invoke({"action": "insert", "cars": resultats_scraping})

'Succès : 20 véhicules insérés dans la base.'

In [88]:
import sqlite3
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats.mstats import winsorize
from langchain_core.tools import tool

@tool
def full_eda_and_viz_tool():
    """
    Outil complet de Data Science : 
    1. Nettoyage (doublons, MCAR/MAR/MNAR) 
    2. Traitement des outliers (Winsorisation)
    3. Visualisation Seaborn/Matplotlib sauvegardée en PNG.
    """
    db_path = "annonces_vehicules.db"
    
    try:
        # 1. Chargement des données
        conn = sqlite3.connect(db_path)
        df_raw = pd.read_sql_query("SELECT * FROM voitures", conn)
        conn.close()

        if df_raw.empty: return "Base de données vide."

        # 2. Nettoyage (Doublons & Valeurs manquantes)
        df = df_raw.drop_duplicates(subset=['title', 'price', 'source']).copy()
        
        # Gestion MCAR/MAR : Imputation par la médiane pour ne pas fausser la moyenne
        df['price'] = df['price'].replace(0, np.nan) # On traite les prix 0 comme manquants
        df['price'] = df['price'].fillna(df['price'].median())
        df['annee'] = df['annee'].fillna(df['annee'].mode()[0])

        # 3. Winsorisation (Outliers)
        # On limite les extrêmes au 5ème et 95ème percentile
        df['price_clean'] = winsorize(df['price'], limits=[0.05, 0.05])
        df['km_clean'] = winsorize(df['kilometrage'], limits=[0.05, 0.05])

        # 4. Visualisation combinée Matplotlib + Seaborn
        sns.set_theme(style="darkgrid")
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Dashboard EDA - Analyse du Marché Automobile (Dakar-Auto)', fontsize=18)

        # A. Distribution Avant/Après (Seaborn)
        sns.kdeplot(df_raw['price'], ax=axes[0,0], label="Brut", fill=True, color="red")
        sns.kdeplot(df['price_clean'], ax=axes[0,0], label="Winsorisé", fill=True, color="green")
        axes[0,0].set_title("Impact du nettoyage sur la Distribution des Prix")
        axes[0,0].legend()

        # B. Boxplot (Outliers)
        sns.boxplot(data=[df_raw['price'], df['price_clean']], ax=axes[0,1], palette="Set2")
        axes[0,1].set_xticklabels(['Brut', 'Winsorisé'])
        axes[0,1].set_title("Comparaison des Outliers (Boxplot)")

        # C. Relation Prix / Kilométrage (Regplot)
        sns.regplot(x='km_clean', y='price_clean', data=df, ax=axes[1,0], scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
        axes[1,0].set_title("Corrélation Prix vs Kilométrage")

        # D. Répartition des Boîtes de vitesse (Countplot)
        sns.countplot(x='boite', data=df, ax=axes[1,1], palette="viridis")
        axes[1,1].set_title("Volume par Type de Boîte")

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        output_image = "dashboard_eda_final.png"
        plt.savefig(output_image)
        plt.close()

        # 5. Rapport de synthèse pour l'Agent
        stats = {
            "total_initial": len(df_raw),
            "total_apres_nettoyage": len(df),
            "prix_moyen_winsorise": round(df['price_clean'].mean(), 2),
            "correlation_prix_km": round(df['price_clean'].corr(df['km_clean']), 2),
            "image_generee": output_image
        }

        return f"Analyse terminée. Rapport : {stats}"

    except Exception as e:
        return f"Erreur lors de l'analyse : {str(e)}"


In [90]:
from sklearn.linear_model import LinearRegression
from langchain_core.tools import tool

@tool
def expert_ml_prediction_tool(annee: int, km: int, carburant: str, boite: str):
    """
    Prédit le prix d'un véhicule en utilisant un modèle entraîné sur la BD.
    Encode automatiquement le carburant et la boîte de vitesse.
    """
    conn = sqlite3.connect("annonces_vehicules.db")
    data = conn.execute("SELECT * FROM voitures").fetchall()
    conn.close()

    if len(data) < 10:
        return "Pas assez de données pour entraîner le modèle IA."

    # 1. Préparation des données (Suppression ID et Source)
    df = pd.DataFrame(data, columns=["ID", "Titre", "Prix", "KM", "Carburant", "Boîte", "Année", "Source"])
    df = df.drop(columns=["ID", "Source"])

    # 2. ENCODAGE (Transformation des textes en nombres)
    # On utilise map pour garder le contrôle sur les valeurs
    df['Carburant_enc'] = df['Carburant'].astype('category').cat.codes
    df['Boite_enc'] = df['Boîte'].astype('category').cat.codes

    # Sauvegarder les correspondances pour l'encodage du véhicule à prédire
    carburant_map = dict(enumerate(df['Carburant'].astype('category').cat.categories))
    boite_map = dict(enumerate(df['Boîte'].astype('category').cat.categories))
    
    # Inverser pour trouver le code depuis le texte
    inv_carburant = {v: k for k, v in carburant_map.items()}
    inv_boite = {v: k for k, v in boite_map.items()}

    # 3. ENTRAÎNEMENT DU MODÈLE (Multi-Linear Regression)
    # Le prix dépend maintenant de 4 facteurs : Année, KM, Carburant, Boîte
    X = df[['Année', 'KM', 'Carburant_enc', 'Boite_enc']]
    y = df['Prix']
    
    model = LinearRegression()
    model.fit(X, y)

    # 4. PRÉDICTION DU VÉHICULE CIBLE
    # On encode les entrées utilisateur
    c_code = inv_carburant.get(carburant, 0)
    b_code = inv_boite.get(boite, 0)
    
    input_data = np.array([[annee, km, c_code, b_code]])
    prix_predit = model.predict(input_data)[0]

    # 5. CALCUL DE L'IMPACT (Pour la soutenance)
    # Quel est l'impact financier d'une boîte auto selon le modèle ?
    impact_boite = model.coef_[3] 

    return {
        "prediction": f"{int(prix_predit):,} FCFA".replace(",", " "),
        "precision_modele_r2": round(model.score(X, y), 2),
        "analyse_decisionnelle": f"Une boîte de type '{boite}' modifie le prix d'environ {int(impact_boite):,} FCFA par rapport à la moyenne.".replace(",", " "),
        "status": "Modèle ML multivarié entraîné avec succès."
    }